In [0]:
import pandas as pd
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import Optional

import pyspark
from pyspark.sql import DataFrame
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType, BooleanType, TimestampType
from pyspark.sql.functions import *
from delta.tables import DeltaTable

@dataclass
class StructAssetTable:
    dataframe: DataFrame
    metadata: Optional[dict] = None
    metrics: Optional[dict] = None

In [0]:
### COMPONENT FOR READING

#INTERFACE AND IMPLEMENTATIONS FOR READING TABLES
class InterfaceTableReader(ABC):

    @abstractmethod
    def read_table(self) -> DataFrame:
        pass

class AbstractFileReader(InterfaceTableReader):

    def __init__(self, file_path: str):
        self.file_path = file_path

    @abstractmethod
    def read_table(self) -> DataFrame:
        pass

class CsvFileReader(AbstractFileReader):

    def __init__(self, file_path: str, separator: str = ','):
        super().__init__(file_path)
        self.separator = separator

    def read_table(self) -> DataFrame:
        return spark.read.format('csv').option('header', 'true').load(self.file_path)
    
#INTERFACE AND IMPLEMENTATIONS FOR READING METADATA
class InterfaceMetadataReader(ABC):

    @abstractmethod
    def read_metadata(self):
        pass
    
class MetadataReader(InterfaceMetadataReader):

    def __init__(self, catalog_name: str, schema_name: str, table_name: str):
        self.catalog_name = catalog_name
        self.schema_name = schema_name
        self.table_name = table_name
        self.metadata = {}

    def read_metadata(self):

        metadata_table = spark.sql(f""" SELECT t.*, s.schema_name
                                        FROM governance_prod.metadata.tables t
                                        INNER JOIN governance_prod.metadata.schemas s ON t.schema_id = s.schema_id
                                        INNER JOIN governance_prod.metadata.catalogs c ON s.catalog_id = c.catalog_id
                                        WHERE table_name = '{self.table_name}' AND current_flag = True
                                            AND s.schema_name = '{self.schema_name}'
                                            AND c.catalog_name = '{self.catalog_name}' LIMIT 1""").collect()
        
        #SET CATALOG, SCHEMA AND ETL MODULE
        table_id = metadata_table[0]['table_id']
        self.metadata['table_id'] = table_id
        self.metadata['catalog_name'] = self.catalog_name
        self.metadata['schema_name'] = self.schema_name
        self.metadata['table_name'] = self.table_name
        self.metadata['etl_module'] = metadata_table[0]['etl_module']
        self.metadata['quality'] = metadata_table[0]['quality']
        self.metadata['write_mode'] = metadata_table[0]['write_mode']
        self.metadata['field_data'] = {}

        #GET TABLE DETAILS  
        metadata_table_detail = spark.sql(f'SELECT * FROM governance_prod.metadata.tables_detail WHERE table_id = {table_id}').collect()

        for row in metadata_table_detail:

            if 'metadata' in row['column_name']:
                continue

            self.metadata['field_data'][row['column_name']] = {'data_type': row['data_type'], 
                                                               'is_nullable': row['is_nullable'], 
                                                               'is_partition': row['is_partition'], 
                                                               'comment': row['comment']}
            
#INTERFACE AND IMPLEMENTATIONS FOR COUNTING ROWS IN TABLES
class InterfaceTableCounter(ABC):

    @abstractmethod
    def get_total_rows(self) -> int:
        pass

class AbstractDeltalakeCounter(InterfaceTableCounter):

    def __init__(self, catalog_name: str, schema_name: str, table_name: str):
        self.catalog_name = catalog_name
        self.schema_name = schema_name
        self.table_name = table_name

    @abstractmethod
    def get_total_rows(self) -> int:
        pass

class DeltalakeCounter(AbstractDeltalakeCounter):

    def get_total_rows(self) -> int:

        complete_table_name = f'{self.catalog_name}.{self.schema_name}.{self.table_name}'
        dataframe = spark.read.table(complete_table_name)
        total_rows = dataframe.count()

        return total_rows

#FACADE FOR TABLE EXTRACTION            
class FacadeTableExtractor():

    def __init__(self, catalog_name, schema_name, table_name, path):
        self.catalog_name = catalog_name
        self.schema_name = schema_name
        self.table_name = table_name
        self.path = path

    def extract(self):

        #EXTRACT METADATA
        metadata_reader = MetadataReader(catalog_name=self.catalog_name, schema_name=self.schema_name, table_name=self.table_name)
        metadata_reader.read_metadata()

        #EXTRACT LANDING FILE
        csv_file_reader = CsvFileReader(file_path=path)
        dataframe = csv_file_reader.read_table()

        self.struct_asset_table = StructAssetTable(dataframe=dataframe, metadata=metadata_reader.metadata)


### COMPONENT FOR WRITING

#INTERFACE AND IMPLEMENTATIONS FOR WRITING TABLES
class InterfaceTableWriter(ABC):

    @abstractmethod
    def write_table(self):
        pass

class AbstractDeltalakeWriter(InterfaceTableWriter):

    def __init__(self, dataframe: DataFrame, catalog_name: str, schema_name: str, table_name):
        self.dataframe = dataframe
        self.catalog_name = catalog_name
        self.schema_name = schema_name
        self.table_name = table_name

    @abstractmethod
    def write_table(self):
        pass

class DeltalakeWriter(AbstractDeltalakeWriter):

    def set_params(self, write_mode: str, batch_id: str):
        self.write_mode = write_mode
        self.batch_id = batch_id

    def write_table(self):
       
        output_table_name = f'{self.catalog_name}.{self.schema_name}.{self.table_name}'
        if self.write_mode == 'overwrite_partition':

            #TO DO - ADD SUPPORT FOR NON DEFAULT PARTITION
            partition_column = 'metadata_batch_id'

            self.dataframe.write.format('delta').mode('overwrite').option('replaceWhere', f"{partition_column} = '{self.batch_id}'").partitionBy(partition_column).saveAsTable(output_table_name)

class DeltalakeMetricsWriter(AbstractDeltalakeWriter):

    def write_table(self):
        
        complete_table_name = f'{self.catalog_name}.{self.schema_name}.{self.table_name}'
        delta_table = DeltaTable.forName(spark, complete_table_name)

        delta_table.alias('target').merge(self.dataframe.alias('source'),
            'target.catalog_name=source.catalog_name AND target.schema_name=source.schema_name AND target.table_name=source.table_name AND target.batch_id=source.batch_id'  
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

class FacadeDeltalakeExporter():

    def __init__(self, struct_asset_table: StructAssetTable):

        self.struct_asset_table = struct_asset_table
        self.accumulated_rows = None
        self.dataframe_ingestion_metrics = None

    def process(self):
        self.write_table()
        self.get_accumulated_rows()
        self.generate_ingestion_metrics()
        self.write_metrics()

    def write_table(self):

        delta_like_writer = DeltalakeWriter(dataframe=self.struct_asset_table.dataframe, 
                                            catalog_name=self.struct_asset_table.metadata['catalog_name'], 
                                            schema_name=self.struct_asset_table.metadata['schema_name'],
                                            table_name=self.struct_asset_table.metadata['table_name'])
        
        delta_like_writer.set_params(write_mode=self.struct_asset_table.metadata['write_mode'], 
                                     batch_id=self.struct_asset_table.metrics['batch_id'])
        
        delta_like_writer.write_table()

    def get_accumulated_rows(self):
        
        delta_lake_counter = DeltalakeCounter(catalog_name=self.struct_asset_table.metadata['catalog_name'], 
                                              schema_name=self.struct_asset_table.metadata['schema_name'], 
                                              table_name=self.struct_asset_table.metadata['table_name'])
        
        self.accumulated_rows = delta_lake_counter.get_total_rows()

    def generate_ingestion_metrics(self):

        columns = ['table_id', 'catalog_name', 'schema_name', 'table_name', 'batch_id', 'loaded_at', 
                   'etl_module', 'write_mode', 'initial_rows', 'final_rows', 'accumulated_rows', 'quality']

        ingestion_metrics = [(self.struct_asset_table.metadata['table_id'],
                            self.struct_asset_table.metadata['catalog_name'], 
                            self.struct_asset_table.metadata['schema_name'], 
                            self.struct_asset_table.metadata['table_name'], 
                            self.struct_asset_table.metrics['batch_id'], 
                            self.struct_asset_table.metrics['loaded_at'], 
                            self.struct_asset_table.metadata['etl_module'], 
                            self.struct_asset_table.metadata['write_mode'], 
                            self.struct_asset_table.metrics['initial_rows'], 
                            self.struct_asset_table.metrics['final_rows'], 
                            self.accumulated_rows,
                            self.struct_asset_table.metadata['quality'])]

        self.dataframe_ingestion_metrics = spark.createDataFrame(ingestion_metrics, columns)

    def write_metrics(self):

        metrics_metadata = {'catalog_name': 'governance', 'schema_name': 'metrics', 'table_name': 'ingestions'}
        struct_asset_metrics = StructAssetTable(dataframe=self.dataframe_ingestion_metrics, metadata=metrics_metadata)

        delta_like_writer = DeltalakeMetricsWriter(dataframe=self.dataframe_ingestion_metrics,
                                                   catalog_name='governance_prod',
                                                   schema_name='metrics',
                                                   table_name='ingestions')
        
        delta_like_writer.write_table()
        
        
## COMPONENT FOR PROCESSING

#INTERFACE AND IMPLEMENTATIONS FOR LAND TO BRONZE PROCESSING
class InterfaceLandToBronzeTemplate(ABC):

    @abstractmethod
    def process(self):
        pass

    @abstractmethod
    def validate_required_columns(self):
        pass

    @abstractmethod
    def cast_data_types(self):
        pass

    @abstractmethod
    def validate_non_nullable_columns(self):
        pass

    @abstractmethod       
    def add_metadata(self):
        pass

    @abstractmethod
    def generate_struct(self):
        pass

class LandToBronzeTemplate(InterfaceLandToBronzeTemplate):

    def __init__(self, struct_asset_table: StructAssetTable, batch_id: str):

        self.dataframe = struct_asset_table.dataframe
        self.metadata = struct_asset_table.metadata
        
        self.metrics = {}
        self.metrics['batch_id'] = batch_id
        self.metrics['loaded_at'] = pd.Timestamp.utcnow().strftime('%Y-%m-%d %H:%M:%S')
        self.metrics['initial_rows'] = self.dataframe.count()
        self.metrics['final_rows'] = None

    def process(self):

        self.validate_required_columns()
        self.cast_data_types()
        self.validate_non_nullable_columns()
        self.add_metadata()
        self.generate_struct()

    def validate_required_columns(self):

        required_columns = self.metadata['field_data'].keys()
        dataframe_columns = self.dataframe.columns

        for column in required_columns:
            if column not in dataframe_columns:
                raise ValueError(f"Missing column in dataframe: {column}")
            
        for column in dataframe_columns:
            if column not in required_columns:
                raise ValueError(f"Missing column in metadata: {column}")

    def cast_data_types(self):
     
        for column in self.metadata['field_data'].keys():

            plain_data_type = self.metadata['field_data'][column]['data_type']
            data_type = None

            if plain_data_type == 'string':
                data_type = StringType()
            elif plain_data_type == 'timestamp':
                data_type = TimestampType()
            elif plain_data_type == 'integer':
                data_type = IntegerType()
            elif plain_data_type == 'boolean':
                data_type = BooleanType()
            else:
                raise ValueError(f"Invalid data type: {plain_data_type}")

            self.dataframe = self.dataframe.withColumn(column, self.dataframe[column].cast(data_type))

    def validate_non_nullable_columns(self):

        for column in self.metadata['field_data'].keys():

            #NON NULLABLE
            if self.metadata['field_data'][column]['is_nullable'] == False:
                count_null = self.dataframe.filter(col(column).isNull()).count()

                if count_null > 0:
                    raise ValueError(f"Null value found in column: {column}") 

    def add_metadata(self):

        #ADD COMMENTS
        for column in self.metadata['field_data'].keys():
            self.dataframe = self.dataframe.withMetadata(column, {'comment': self.metadata['field_data'][column]['comment']})

        #ADD METADATA COLUMNS
        self.dataframe = self.dataframe.withColumn('metadata_batch_id', lit(self.metrics['batch_id']))
        self.dataframe = self.dataframe.withColumn('metadata_loaded_at', lit(self.metrics['loaded_at']))
        self.dataframe = self.dataframe.withColumn('metadata_etl_module', lit(self.metadata['etl_module']))

        self.dataframe = self.dataframe.withColumn('metadata_loaded_at', to_timestamp(col('metadata_loaded_at'), "yyyy-MM-dd HH:mm:ss"))

        self.dataframe = self.dataframe.withMetadata('metadata_batch_id', {'comment': 'Identifier of the batch process'})
        self.dataframe = self.dataframe.withMetadata('metadata_loaded_at', {'comment': 'Timestamp when the record was loaded into the table'})
        self.dataframe = self.dataframe.withMetadata('metadata_etl_module', {'comment': 'ETL module used to process this record'})        

    def generate_struct(self):

        self.metrics['final_rows'] = self.dataframe.count()        
        self.struct_asset_table = StructAssetTable(metadata=self.metadata, dataframe=self.dataframe, metrics=self.metrics)

In [0]:
domain = 'medallion'
environment = 'dev'
batch_id = '20251101_000000'
path = f"dbfs:/Workspace/Users/armando.n90@gmail.com/databricks_case/lakehouse/landing/healthsys/{batch_id}/cat_cie_10.csv"
schema_name = 'bronze_healthsys'
table_name = 'raw_cat_cie_10'
catalog_name = domain + '_' + environment

In [0]:
facade_table_extractor = FacadeTableExtractor(catalog_name=catalog_name, schema_name=schema_name, table_name=table_name, path=path)
facade_table_extractor.extract()

extractor_asset = facade_table_extractor.struct_asset_table

print(extractor_asset.metadata)
extractor_asset.dataframe.show(5)

In [0]:
table_processor = LandToBronzeTemplate(struct_asset_table=extractor_asset, batch_id=batch_id)   
table_processor.process()

processor_asset = table_processor.struct_asset_table

print(processor_asset.metadata)
print(processor_asset.metrics)
processor_asset.dataframe.show(5)

In [0]:
facade_deltalake_exporter = FacadeDeltalakeExporter(struct_asset_table=processor_asset)
facade_deltalake_exporter.process()

In [0]:
#test = spark.read.table('medallion_dev.bronze_healthsys.raw_cat_cie_10')
test = spark.read.table('governance_prod.metrics.ingestions')
print(test.count())
test.show(5)


In [0]:
#spark.sql('DROP TABLE IF EXISTS medallion_dev.bronze_healthsys.raw_cat_cie_10')
#spark.sql('DELETE FROM governance.metrics.ingestions')

In [0]:
#todo arreglar identificadores en esquemas